In [1]:
!pip install SPARQLWrapper
!pip install sparql-dataframe

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import sparql_dataframe

endpoint = "http://localhost:7200/repositories/mqtt4ssn"

In [3]:
NS = {
    "mqtt:": "https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#",
    "rdfs:": "http://www.w3.org/2000/01/rdf-schema#",
    "ex:": "http://example.org/",
    "sosa": "http://www.w3.org/ns/sosa/" 
}

In [4]:
import pandas as pd

def strip_namespaces(df, ns_map=NS):
    def shrink(val):
        if not isinstance(val, str):
            return val
        # ersetze bekannte Namespaces durch Prefix
        for pref, base in ns_map.items():
            if val.startswith(base):
                return pref + val[len(base):]
        # fallback: nur local name nach # oder /
        if "#" in val:
            return val.rsplit("#", 1)[-1]
        if "/" in val:
            return val.rsplit("/", 1)[-1]
        return val

    return df.map(shrink)


## Application Message

### CQ2.01u - Which Application Message encodes which SOSA Activity?

In [5]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX sosa: <http://www.w3.org/ns/sosa/> 

SELECT ?message ?activity ?member ?FOI ?property WHERE {
  ?message a mqtt:ApplicationMessage ;
          ?applicationMessageEncodesActivity ?activity .
  ?activity a mqtt:Activity .
  OPTIONAL {?activity sosa:hasFeatureOfInterest ?FOI}
  OPTIONAL {?activity sosa:hasMember ?member}
  OPTIONAL {?member sosa:observedProperty ?property}
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,message,activity,member,FOI,property
0,ex:ApplicationMessage_1764845861738,ex:ObservationCollection_RFID_87A3F1_2025-12-0...,ex:Observation_GripPoint_X,ex:FeatureOfInterest_RFID_87A3F1,ex:Property_GripPoint_X
1,ex:ApplicationMessage_1764845861738,ex:ObservationCollection_RFID_87A3F1_2025-12-0...,ex:Observation_GripPoint_Y,ex:FeatureOfInterest_RFID_87A3F1,ex:Property_GripPoint_Y
2,ex:ApplicationMessage_1764845861738,ex:ObservationCollection_RFID_87A3F1_2025-12-0...,ex:Observation_GripPoint_Z,ex:FeatureOfInterest_RFID_87A3F1,ex:Property_GripPoint_Z
3,ex:ApplicationMessage_1764845861738,ex:ObservationCollection_RFID_87A3F2_2025-12-0...,ex:Observation_GripPoint_X,ex:FeatureOfInterest_RFID_87A3F2,ex:Property_GripPoint_X
4,ex:ApplicationMessage_1764845861738,ex:ObservationCollection_RFID_87A3F2_2025-12-0...,ex:Observation_GripPoint_Y,ex:FeatureOfInterest_RFID_87A3F2,ex:Property_GripPoint_Y
5,ex:ApplicationMessage_1764845861738,ex:ObservationCollection_RFID_87A3F2_2025-12-0...,ex:Observation_GripPoint_Z,ex:FeatureOfInterest_RFID_87A3F2,ex:Property_GripPoint_Z
6,ex:ApplicationMessage_1764845861738,ex:ObservationCollection_RFID_87A3F3_2025-12-0...,ex:Observation_GripPoint_X,ex:FeatureOfInterest_RFID_87A3F3,ex:Property_GripPoint_X
7,ex:ApplicationMessage_1764845861738,ex:ObservationCollection_RFID_87A3F3_2025-12-0...,ex:Observation_GripPoint_Y,ex:FeatureOfInterest_RFID_87A3F3,ex:Property_GripPoint_Y
8,ex:ApplicationMessage_1764845861738,ex:ObservationCollection_RFID_87A3F3_2025-12-0...,ex:Observation_GripPoint_Z,ex:FeatureOfInterest_RFID_87A3F3,ex:Property_GripPoint_Z
9,ex:ApplicationMessage_1764845861738,ex:ObservationCollection_RFID_87A3F1_2025-12-0...,ex:Observation_GripPoint_X,ex:FeatureOfInterest_RFID_87A3F1,ex:Property_GripPoint_X


### CQ2.02u - Which fields are encoded in the Application Message?

In [6]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?message ?fields WHERE {
  ?message a mqtt:ApplicationMessage ;
          mqtt:hasEncodedFields ?fields .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,message,fields
0,ex:ApplicationMessage_1764845861738,"RFID,JobStart,JobEnd,GripPoint_X,GripPoint_Y,G..."
1,ex:ApplicationMessage_1764845868422,"RFID,JobStart,JobEnd,Correction_X,Correction_Y..."


### CQ2.03u - What content, content type, encoding and delimiter does an Application Message have?

In [7]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?message ?content ?type ?encoding ?delimiter WHERE {
  ?message a mqtt:ApplicationMessage ;
  OPTIONAL { ?message mqtt:hasContent ?content }
  OPTIONAL { ?message mqtt:hasContentType ?type }
  OPTIONAL { ?message mqtt:hasEncoding ?encoding }
  OPTIONAL { ?message mqtt:hasContentDelimiter ?delimiter }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,message,content,type,encoding,delimiter
0,ex:ApplicationMessage_1764845861738,RFID;JobStart;JobEnd;GripPoint_X;GripPoint_Y;G...,csv,NaN,NaN
1,ex:PublishPayload_1764845861738,RFID;JobStart;JobEnd;GripPoint_X;GripPoint_Y;G...,NaN,NaN,NaN
2,ex:PublishProperties_1764845861738,NaN,csv,NaN,NaN
3,ex:ApplicationMessage_1764845868422,RFID;JobStart;JobEnd;Correction_X;Correction_Y...,csv,NaN,NaN
4,ex:PublishPayload_1764845868422,RFID;JobStart;JobEnd;Correction_X;Correction_Y...,NaN,NaN,NaN
5,ex:PublishProperties_1764845868422,NaN,csv,NaN,NaN


### CQ2.04u - Who has send which Application Messages?

In [8]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?message ?payload ?packet ?client ?broker WHERE {
  ?message a mqtt:ApplicationMessage ;
           mqtt:isApplicationMessageOf ?payload .
  ?payload mqtt:isPublishPayloadOf ?packet .
  OPTIONAL { ?client mqtt:sendsControlPacket ?packet . ?client a mqtt:Client }
  OPTIONAL { ?broker mqtt:sendsControlPacket ?packet . ?broker a mqtt:Broker }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,message,payload,packet,client,broker
0,ex:ApplicationMessage_1764845861738,ex:PublishPayload_1764845861738,ex:PublishPacket_1764845861738,ex:Broker_emqx%40172.19.0.3,ex:Broker_emqx%40172.19.0.3
1,ex:ApplicationMessage_1764845861738,ex:PublishPayload_1764845861738,ex:PublishPacket_1764845861738,ex:Client_Client%201,ex:Broker_emqx%40172.19.0.3
2,ex:ApplicationMessage_1764845868422,ex:PublishPayload_1764845868422,ex:PublishPacket_1764845868422,ex:Broker_emqx%40172.19.0.3,ex:Broker_emqx%40172.19.0.3
3,ex:ApplicationMessage_1764845868422,ex:PublishPayload_1764845868422,ex:PublishPacket_1764845868422,ex:Client_Client%201,ex:Broker_emqx%40172.19.0.3
